# Baseline Retrieval

## Objective
This notebook aims to create a straightforward, transparent baseline for generating top-N item candidates before incorporating conversational parsing, reranking, or retrieval-augmented generation (RAG). Insights from `01_catalog_and_reviews_eda.ipynb` reveal that item metadata alone already provides a strong foundation for baseline retrieval, without relying on the full review corpus. The focus here is on building an easy-to-understand candidate generator. Later notebooks will evaluate its performance, introduce user preference parsing, ground candidates in review evidence, and develop methods for reranking and interpreting retrieved items.

## Inputs
- `../data/items.parquet`
- `../data/reviews.parquet`

## Outputs
- a lightweight metadata-based retrieval index
- reusable helper functions for item-to-item and short-query retrieval
- example top-N recommendations
- lightweight artifacts for later notebooks

## Baseline Objective and Design Choice

The retrieval baseline uses a hybrid approach combining BM25 (sparse lexical) and dense embeddings (BAAI/bge-small-en-v1.5) fused with Reciprocal Rank Fusion (RRF). This captures both exact-term matches and semantic similarity.

Two entry points are supported:

- item-to-item retrieval: "find items similar to item X" (dense FAISS search)
- short-query retrieval: hybrid BM25 + dense for best recall

To stay within the notebook memory budget, the index is built on a reproducible catalog of up to 300k items per source, ordered by `rating_number` and `average_rating`. That makes the baseline practical to run while still covering the three project domains.

In [1]:
import json
import os
import pickle
import sys
from functools import partial
from pathlib import Path

import torch

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-crs")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import duckdb
import faiss
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from IPython.display import display

PROJECT_ROOT_CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
for candidate in PROJECT_ROOT_CANDIDATES:
    if (candidate / "functions").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the project root containing functions/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from functions.io import scalar as shared_scalar, sql as shared_sql
from functions.retrieval import (
    hybrid_search as shared_hybrid_search,
    search_similar as shared_search_similar,
    tokenize_for_bm25,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR_CANDIDATES = [Path("../data").resolve(), Path("data").resolve()]
for candidate in DATA_DIR_CANDIDATES:
    if (candidate / "items.parquet").exists() and (candidate / "reviews.parquet").exists():
        DATA_DIR = candidate
        break
else:
    raise FileNotFoundError("Could not find items.parquet and reviews.parquet in ../data or data.")

In [2]:
design_summary = pd.DataFrame(
    [
        {
            "component": "index unit",
            "choice": "parent item (`parent_asin`)",
            "reason": "Matches the cleaned catalog grain from the acquisition notebook.",
        },
        {
            "component": "representation",
            "choice": "title + store + category + taxonomy path",
            "reason": "Keeps the query-facing text short and interpretable while preserving the strongest product-type signals.",
        },
        {
            "component": "candidate pool",
            "choice": f"up to {ITEMS_PER_SOURCE:,} items per source",
            "reason": "Keeps the working catalog under the pandas safety cap.",
        },
        {
            "component": "retrieval model",
            "choice": "BM25 + dense embeddings (BAAI/bge-small-en-v1.5) fused via RRF",
            "reason": "BM25 handles exact-term queries; dense embeddings handle semantic queries. RRF merges both without score normalization.",
        },
        {
            "component": "supported tasks",
            "choice": "similar-item retrieval and short text-query retrieval",
            "reason": "Useful enough to seed downstream reranking and grounding notebooks.",
        },
        {
            "component": "price policy",
            "choice": "exclude `price = null` from retrieval pool",
            "reason": PRICE_POLICY_DESCRIPTION,
        },
        {
            "component": "reviews usage",
            "choice": "not used in the retrieval index",
            "reason": "Review grounding is deferred to notebook 04.",
        },
    ]
)

display(design_summary)

,component,choice,reason
0,index unit,parent item (`parent_asin`),Matches the cleaned catalog grain from the acquisition notebook.
1,representation,title + store + category + taxonomy path,Keeps the query-facing text short and interpretable while preserving the strongest product-type signals.
2,candidate pool,"up to 100,000 items per source",Keeps the working catalog under the pandas safety cap.
3,retrieval model,BM25 + dense embeddings (BAAI/bge-small-en-v1.5) fused via RRF,BM25 handles exact-term queries; dense embeddings handle semantic queries. RRF merges both without score normalization.
4,supported tasks,similar-item retrieval and short text-query retrieval,Useful enough to seed downstream reranking and grounding notebooks.
5,price policy,exclude `price = null` from retrieval pool,Exclude items with null price from the retrievable pool because they were likely not purchasable when the snapshot was collected.
6,reviews usage,not used in the retrieval index,Review grounding is deferred to notebook 04.


## Safe Data Access

Before building anything, confirm the real schemas and restrict the notebook to the columns that actually exist. The retrieval index itself only uses item metadata, but the review schema is still inspected because later notebooks will attach review-grounded evidence to the candidates generated here.
    

In [3]:
items_schema = sql("DESCRIBE SELECT * FROM items")
reviews_schema = sql("DESCRIBE SELECT * FROM reviews")
source_count = int(scalar("SELECT COUNT(DISTINCT source) FROM items"))
assert (
    ITEMS_PER_SOURCE * source_count <= MAX_PANDAS_ROWS
), "Candidate pool would exceed the pandas safety cap."

retrieval_columns = pd.DataFrame(
    [
        {"dataset": "items", "column": "parent_asin", "used_in_baseline": True, "purpose": "item key"},
        {
            "dataset": "items",
            "column": "source",
            "used_in_baseline": True,
            "purpose": "domain filter and same-source retrieval constraint",
        },
        {
            "dataset": "items",
            "column": "main_category",
            "used_in_baseline": True,
            "purpose": "high-level lexical context",
        },
        {
            "dataset": "items",
            "column": "title",
            "used_in_baseline": True,
            "purpose": "primary lexical signal",
        },
        {"dataset": "items", "column": "store", "used_in_baseline": True, "purpose": "brand/store signal"},
        {
            "dataset": "items",
            "column": "categories_path",
            "used_in_baseline": True,
            "purpose": "taxonomy context",
        },
        {
            "dataset": "items",
            "column": "description_text",
            "used_in_baseline": False,
            "purpose": "reserved for later grounding / evidence extraction",
        },
        {
            "dataset": "items",
            "column": "features_text",
            "used_in_baseline": False,
            "purpose": "reserved for later grounding / evidence extraction",
        },
        {
            "dataset": "items",
            "column": "average_rating",
            "used_in_baseline": True,
            "purpose": "tie-break display / popularity prior",
        },
        {
            "dataset": "items",
            "column": "rating_number",
            "used_in_baseline": True,
            "purpose": "pool selection and tie-breaks",
        },
        {
            "dataset": "items",
            "column": "price",
            "used_in_baseline": True,
            "purpose": "candidate availability filter and result inspection",
        },
        {
            "dataset": "reviews",
            "column": "title / text",
            "used_in_baseline": False,
            "purpose": "reserved for later grounding notebooks",
        },
        {
            "dataset": "reviews",
            "column": "rating / helpful_vote / verified_purchase",
            "used_in_baseline": False,
            "purpose": "reserved for evidence weighting later",
        },
    ]
)

display(items_schema)
display(reviews_schema)
display(retrieval_columns)

,column_name,column_type,null,key,default,extra
0,parent_asin,VARCHAR,YES,None,None,None
1,main_category,VARCHAR,YES,None,None,None
2,title,VARCHAR,YES,None,None,None
3,store,VARCHAR,YES,None,None,None
4,source,VARCHAR,YES,None,None,None
5,average_rating,DOUBLE,YES,None,None,None
6,rating_number,BIGINT,YES,None,None,None
7,price,DOUBLE,YES,None,None,None
8,categories_path,VARCHAR,YES,None,None,None
9,description_text,VARCHAR,YES,None,None,None


,column_name,column_type,null,key,default,extra
0,rating,DOUBLE,YES,None,None,None
1,title,VARCHAR,YES,None,None,None
2,text,VARCHAR,YES,None,None,None
3,asin,VARCHAR,YES,None,None,None
4,parent_asin,VARCHAR,YES,None,None,None
5,user_id,VARCHAR,YES,None,None,None
6,timestamp,BIGINT,YES,None,None,None
7,helpful_vote,BIGINT,YES,None,None,None
8,verified_purchase,BOOLEAN,YES,None,None,None
9,source,VARCHAR,YES,None,None,None


,dataset,column,used_in_baseline,purpose
0,items,parent_asin,True,item key
1,items,source,True,domain filter and same-source retrieval constraint
2,items,main_category,True,high-level lexical context
3,items,title,True,primary lexical signal
4,items,store,True,brand/store signal
5,items,categories_path,True,taxonomy context
6,items,description_text,False,reserved for later grounding / evidence extraction
7,items,features_text,False,reserved for later grounding / evidence extraction
8,items,average_rating,True,tie-break display / popularity prior
9,items,rating_number,True,pool selection and tie-breaks


In [4]:
candidate_pool_query = r"""
WITH eligible AS (
    SELECT
        parent_asin,
        source,
        main_category,
        title,
        store,
        average_rating,
        rating_number,
        price,
        trim(regexp_replace(regexp_replace(coalesce(main_category, ''), '[\[\]"]', ' ', 'g'), '\s+', ' ', 'g')) AS main_category_clean,
        trim(regexp_replace(regexp_replace(coalesce(title, ''), '[\[\]"]', ' ', 'g'), '\s+', ' ', 'g')) AS title_clean,
        trim(regexp_replace(regexp_replace(coalesce(store, ''), '[\[\]"]', ' ', 'g'), '\s+', ' ', 'g')) AS store_clean,
        left(trim(regexp_replace(regexp_replace(coalesce(categories_path, ''), '[\[\]"]', ' ', 'g'), '\s+', ' ', 'g')), 400) AS categories_clean,
        left(trim(regexp_replace(regexp_replace(coalesce(description_text, ''), '[\[\]"]', ' ', 'g'), '\s+', ' ', 'g')), 800) AS description_clean,
        left(trim(regexp_replace(regexp_replace(coalesce(features_text, ''), '[\[\]"]', ' ', 'g'), '\s+', ' ', 'g')), 800) AS features_clean
    FROM items
    WHERE title IS NOT NULL
      AND trim(title) <> ''
      AND (
        (categories_path IS NOT NULL AND trim(categories_path) NOT IN ('', '[]'))
        OR (description_text IS NOT NULL AND trim(description_text) NOT IN ('', '[]'))
        OR (features_text IS NOT NULL AND trim(features_text) NOT IN ('', '[]'))
      )
      AND ({price_filter_sql})
), ranked AS (
    SELECT
        *,
        row_number() OVER (
            PARTITION BY source
            ORDER BY coalesce(rating_number, 0) DESC, coalesce(average_rating, 0) DESC, parent_asin
        ) AS source_rank
    FROM eligible
)
SELECT
    parent_asin,
    source,
    main_category,
    title,
    store,
    average_rating,
    rating_number,
    price,
    concat_ws(
        ' ',
        title_clean,
        title_clean,
        store_clean,
        main_category_clean,
        categories_clean
    ) AS retrieval_text,
    length(
        concat_ws(
            ' ',
            title_clean,
            title_clean,
            store_clean,
            main_category_clean,
            categories_clean
        )
    ) AS retrieval_text_chars
FROM ranked
WHERE source_rank <= {items_per_source}
ORDER BY source, source_rank
""".format(
    items_per_source=ITEMS_PER_SOURCE, price_filter_sql=PRICE_FILTER_SQL
)

candidate_pool = sql(candidate_pool_query).reset_index(drop=True)
candidate_pool["retrieval_row_id"] = np.arange(len(candidate_pool), dtype=np.int32)
assert len(candidate_pool) <= MAX_PANDAS_ROWS, "Candidate pool exceeded the pandas safety cap."

candidate_pool_summary = candidate_pool.groupby("source", as_index=False).agg(
    items=("parent_asin", "size"),
    median_rating_number=("rating_number", "median"),
    avg_retrieval_text_chars=("retrieval_text_chars", "mean"),
)

memory_mb = candidate_pool.memory_usage(deep=True).sum() / (1024**2)
print(f"Candidate pool rows: {len(candidate_pool):,}")
print(f"Candidate pool memory footprint: {memory_mb:.2f} MB")
print(f"Price policy: {PRICE_POLICY_DESCRIPTION}")
display(candidate_pool_summary)
display(
    candidate_pool[["source", "parent_asin", "title", "rating_number", "average_rating", "price"]].head(5)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Candidate pool rows: 300,000
Candidate pool memory footprint: 177.98 MB
Price policy: Exclude items with null price from the retrievable pool because they were likely not purchasable when the snapshot was collected.


,source,items,median_rating_number,avg_retrieval_text_chars
0,electronics,100000,442.0,389.97603
1,home_and_kitchen,100000,1069.0,379.58818
2,sports_and_outdoors,100000,326.0,319.70876


,source,parent_asin,title,rating_number,average_rating,price
0,electronics,B07KTYJ769,"Amazon Smart Plug, for home automation, Works with Alexa - A Certified for Humans Device",542825,4.7,24.99
1,electronics,B0BGNG1294,"Amazon Basics HDMI Cable, 18Gbps High-Speed, 4K@60Hz, 2160p, Ethernet Ready, 10 Foot, Black",507202,4.7,8.55
2,electronics,B08CLNX58K,SanDisk 32GB 2-Pack Ultra MicroSDHC UHS-I Memory Card (2x32GB) - SDSQUAR-032G-GN6MT,402800,4.7,13.40
3,electronics,B08WJSHSLC,Fire TV Stick (3rd Gen) with Alexa Voice Remote (includes TV controls) + Star Wars The Mandalorian remote cover (Grogu Green),379565,4.7,44.98
4,electronics,B07HZLHPKP,"Echo Show 5 (1st Gen, 2019 release) -- Smart display with Alexa – stay connected with video calling - Charcoal",356248,4.6,79.99


## Candidate Representation

The retrieval text is plain and inspectable. The title is repeated once to increase its influence, while store, main category, and taxonomy path provide supporting lexical context. Long description/features were dropped from the retrieval string because they introduced too much lexical noise for short product queries; those richer fields are better reserved for later grounding and explanation stages.
    

In [5]:
representation_fields = pd.DataFrame(
    [
        {
            "field": "title",
            "included": True,
            "notes": "Primary lexical anchor; repeated twice in the concatenated text.",
        },
        {
            "field": "store",
            "included": True,
            "notes": "Adds brand/store cues when users mention them indirectly.",
        },
        {"field": "main_category", "included": True, "notes": "Adds coarse domain context."},
        {"field": "categories_path", "included": True, "notes": "Adds more specific taxonomy terms."},
        {
            "field": "description_text",
            "included": False,
            "notes": "Reserved for later grounding because it added too much noise for short retrieval queries.",
        },
        {
            "field": "features_text",
            "included": False,
            "notes": "Reserved for later grounding because long bullet text diluted lexical matching.",
        },
        {
            "field": "review text",
            "included": False,
            "notes": "Deferred to later notebooks to keep the baseline lightweight.",
        },
    ]
)

text_length_summary = (
    candidate_pool["retrieval_text_chars"].describe(percentiles=[0.5, 0.9, 0.99]).to_frame().T
)
representation_preview = (
    candidate_pool[["source", "parent_asin", "title", "retrieval_text_chars", "retrieval_text"]]
    .head(3)
    .copy()
)
representation_preview["retrieval_text"] = representation_preview["retrieval_text"].str.slice(0, 280) + " ..."

display(representation_fields)
display(text_length_summary)
display(representation_preview)

,field,included,notes
0,title,True,Primary lexical anchor; repeated twice in the concatenated text.
1,store,True,Adds brand/store cues when users mention them indirectly.
2,main_category,True,Adds coarse domain context.
3,categories_path,True,Adds more specific taxonomy terms.
4,description_text,False,Reserved for later grounding because it added too much noise for short retrieval queries.
5,features_text,False,Reserved for later grounding because long bullet text diluted lexical matching.
6,review text,False,Deferred to later notebooks to keep the baseline lightweight.


,count,mean,std,min,50%,90%,99%,max
retrieval_text_chars,300000.0,363.09099,112.767078,41.0,375.0,502.0,551.0,1343.0


,source,parent_asin,title,retrieval_text_chars,retrieval_text
0,electronics,B07KTYJ769,"Amazon Smart Plug, for home automation, Works with Alexa - A Certified for Humans Device",200,"Amazon Smart Plug, for home automation, Works with Alexa - A Certified for Humans Device Amazon Smart Plug, for home automation, Works with Alexa - A Certified for Humans Device Amazon Amazon Devi..."
1,electronics,B0BGNG1294,"Amazon Basics HDMI Cable, 18Gbps High-Speed, 4K@60Hz, 2160p, Ethernet Ready, 10 Foot, Black",288,"Amazon Basics HDMI Cable, 18Gbps High-Speed, 4K@60Hz, 2160p, Ethernet Ready, 10 Foot, Black Amazon Basics HDMI Cable, 18Gbps High-Speed, 4K@60Hz, 2160p, Ethernet Ready, 10 Foot, Black Amazon Basic..."
2,electronics,B08CLNX58K,SanDisk 32GB 2-Pack Ultra MicroSDHC UHS-I Memory Card (2x32GB) - SDSQUAR-032G-GN6MT,292,SanDisk 32GB 2-Pack Ultra MicroSDHC UHS-I Memory Card (2x32GB) - SDSQUAR-032G-GN6MT SanDisk 32GB 2-Pack Ultra MicroSDHC UHS-I Memory Card (2x32GB) - SDSQUAR-032G-GN6MT SanDisk Computers Electronic...


## Retrieval Baseline Implementation

The hybrid index combines BM25 (lexical matching) with dense embeddings (semantic similarity), fused via Reciprocal Rank Fusion. BM25 handles exact-term queries like brand names and SKUs; the dense encoder handles semantic queries like "comfortable for travel". RRF merges the two ranked lists without requiring score normalization.

In [6]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
bm25_path = ARTIFACT_DIR / "bm25_index.pkl"
embeddings_path = ARTIFACT_DIR / "item_embeddings.npy"
faiss_path = ARTIFACT_DIR / "faiss_index.bin"
pool_path_cached = ARTIFACT_DIR / "candidate_pool.parquet"
metadata_path = ARTIFACT_DIR / "index_metadata.json"

artifacts_exist = (
    bm25_path.exists()
    and embeddings_path.exists()
    and faiss_path.exists()
    and pool_path_cached.exists()
    and metadata_path.exists()
)

artifacts_compatible = False
if artifacts_exist:
    saved_metadata = json.loads(metadata_path.read_text())
    artifacts_compatible = (
        saved_metadata.get("baseline_name") == BASELINE_NAME
        and saved_metadata.get("bm25_tokenizer") == "regex_alnum_lower"
    )

if artifacts_exist and artifacts_compatible and not FORCE_REBUILD:
    # ── Load pre-built indexes + saved candidate_pool (guarantees row alignment) ──
    print("Loading pre-built indexes from disk...")
    candidate_pool = pd.read_parquet(pool_path_cached).reset_index(drop=True)
    print(f"  candidate_pool: {len(candidate_pool):,} rows (from saved parquet)")

    with open(bm25_path, "rb") as f:
        bm25_index = pickle.load(f)
    print(f"  BM25: {bm25_index.corpus_size:,} docs loaded")

    encoder = SentenceTransformer(ENCODER_MODEL, local_files_only=True)

    faiss_index = faiss.read_index(str(faiss_path))
    print(f"  FAISS: {faiss_index.ntotal:,} vectors loaded")

    item_embeddings = np.load(embeddings_path, mmap_mode="r")
    print(f"  Embeddings (mmap): {item_embeddings.shape}")

    assert len(candidate_pool) == faiss_index.ntotal == item_embeddings.shape[0], (
        f"Artifact mismatch: pool={len(candidate_pool)}, "
        f"faiss={faiss_index.ntotal}, embeddings={item_embeddings.shape[0]}. "
        "Set FORCE_REBUILD=True to rebuild."
    )

else:
    if FORCE_REBUILD:
        print("FORCE_REBUILD=True — rebuilding indexes from scratch...")
    elif artifacts_exist and not artifacts_compatible:
        print("Artifacts found but metadata is incompatible with the current baseline — rebuilding...")
    else:
        print("Artifacts not found — building indexes from scratch...")

    # ── 1. BM25 ───────────────────────────────────────────────────────────
    tokenized_corpus = [tokenize_for_bm25(text) for text in candidate_pool["retrieval_text"]]
    bm25_index = BM25Okapi(tokenized_corpus)
    del tokenized_corpus
    print(f"  BM25: {bm25_index.corpus_size:,} docs")
    with open(bm25_path, "wb") as f:
        pickle.dump(bm25_index, f)
    print("  bm25_index.pkl saved")

    # ── 2. Dense embeddings ────────────────────────────────────────────────
    encoder = SentenceTransformer(ENCODER_MODEL, local_files_only=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"  Using device: {device}")
    item_embeddings = encoder.encode(
        candidate_pool["retrieval_text"].tolist(),
        batch_size=ENCODER_BATCH_SIZE,
        device=device,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    faiss.normalize_L2(item_embeddings)
    print(f"  Embeddings: {item_embeddings.shape}")
    np.save(embeddings_path, item_embeddings)
    print("  item_embeddings.npy saved")

    # ── 3. FAISS ──────────────────────────────────────────────────────────
    faiss_index = faiss.IndexFlatIP(EMBEDDING_DIM)
    faiss_index.add(item_embeddings)
    print(f"  FAISS: {faiss_index.ntotal:,} vectors")
    faiss.write_index(faiss_index, str(faiss_path))
    print("  faiss_index.bin saved")

    # ── 4. Reload embeddings as memory-mapped to free RAM ──────────────────
    del item_embeddings
    item_embeddings = np.load(embeddings_path, mmap_mode="r")
    print(f"  item_embeddings reloaded as mmap: {item_embeddings.shape}")

asin_to_row = dict(zip(candidate_pool["parent_asin"], candidate_pool.index))
source_to_row_ids = {
    str(source).lower(): frame.index.to_numpy(dtype=np.int32)
    for source, frame in candidate_pool.groupby("source", sort=False)
}

artifacts = {
    "candidate_pool": candidate_pool,
    "bm25_index": bm25_index,
    "faiss_index": faiss_index,
    "item_embeddings": item_embeddings,
    "encoder": encoder,
    "asin_to_row": asin_to_row,
    "source_to_row_ids": source_to_row_ids,
    "index_metadata": {"hybrid_pool_size": HYBRID_RETRIEVAL_POOL, "rrf_k": RRF_K},
}

hybrid_search = partial(shared_hybrid_search, artifacts=artifacts, pool_size=HYBRID_RETRIEVAL_POOL)
search_similar = partial(shared_search_similar, artifacts=artifacts)

index_stats = pd.DataFrame(
    [
        {
            "candidate_items": len(candidate_pool),
            "embedding_dim": item_embeddings.shape[1],
            "faiss_index_type": "IndexFlatIP",
            "bm25_avg_doc_tokens": round(bm25_index.avgdl, 1),
            "encoder_model": ENCODER_MODEL,
        }
    ]
)
display(index_stats)

Artifacts found but metadata is incompatible with the current baseline — rebuilding...
  BM25: 300,000 docs
  bm25_index.pkl saved


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Using device: cuda


Batches:   0%|          | 0/1172 [00:00<?, ?it/s]

  Embeddings: (300000, 384)
  item_embeddings.npy saved
  FAISS: 300,000 vectors
  faiss_index.bin saved
  item_embeddings reloaded as mmap: (300000, 384)


,candidate_items,embedding_dim,faiss_index_type,bm25_avg_doc_tokens,encoder_model
0,300000,384,IndexFlatIP,53.5,BAAI/bge-small-en-v1.5


In [7]:
RESULT_COLUMNS = [
    "retrieval_row_id",
    "parent_asin",
    "source",
    "main_category",
    "title",
    "rating_number",
    "average_rating",
    "price",
    "retrieval_score",
]

## Example Recommendations

The examples below are only sanity checks, but they are useful ones. The goal is not to prove that the baseline is optimal. The goal is to confirm that a transparent lexical index can already return coherent candidates that later notebooks can rerank, ground, and explain.
    

In [8]:
example_anchors = (
    candidate_pool.sort_values(["source", "rating_number", "average_rating"], ascending=[True, False, False])
    .groupby("source", as_index=False)
    .head(1)[["source", "parent_asin", "title", "rating_number", "average_rating"]]
    .reset_index(drop=True)
)

display(example_anchors)

for row in example_anchors.itertuples(index=False):
    print(f"\nAnchor item ({row.source}): {row.title} [{row.parent_asin}]")
    display(search_similar(row.parent_asin, top_k=5, source_filter=row.source))

,source,parent_asin,title,rating_number,average_rating
0,electronics,B07KTYJ769,"Amazon Smart Plug, for home automation, Works with Alexa - A Certified for Humans Device",542825,4.7
1,home_and_kitchen,B00U8QEXBS,"Amazon Basics Lightweight Super Soft Easy Care Microfiber Bed Sheet Set with Two Pillowcases - Twin, Bright White",369516,4.6
2,sports_and_outdoors,B08D9FWVJD,"IRON °FLASK Sports Water Bottle - 32oz, 3 Lids (Straw Lid), Leak Proof - Stainless Steel Gym & Sport Bottles for Men, Women & Kids - Double Walled, Insulated Thermos, Metal Canteen",121670,4.8



Anchor item (electronics): Amazon Smart Plug, for home automation, Works with Alexa - A Certified for Humans Device [B07KTYJ769]


,retrieval_row_id,parent_asin,source,main_category,title,rating_number,average_rating,price,retrieval_score
0,9033,B09DF5NBZB,electronics,Amazon Devices,"Amazon Smart Air Quality Monitor – Know your air, Works with Alexa– A Certified for Humans Device",3110,4.2,69.99,0.838989
1,66629,B01MQHCNVR,electronics,Tools & Home Improvement,"Samsung SmartThings Outlet, Works with Amazon Alexa",296,4.1,41.75,0.819896
2,292,B07RR5WZCR,electronics,Amazon Devices,"Echo Glow - Multicolor smart lamp, a Certified for Humans Device – Requires compatible Alexa device",45248,4.6,19.99,0.808958
3,25316,B0BTHGCS1D,electronics,Tools & Home Improvement,"Smart Plug for Alexa Devices, Smart Outlet Bluetooth Mesh, Simple Set Up, Voice and Alexa App Remote Control, ETL & FCC Certified, 4 Pack",1042,4.1,39.99,0.801136
4,22696,B00EOEDJ9W,electronics,Cell Phones & Accessories,"Wemo Insight WiFi Enabled Smart Plug, with Energy Monitoring, Works with Alexa (Discontinued by Manufacturer - Newer Version Available)",1184,3.7,29.99,0.786590



Anchor item (home_and_kitchen): Amazon Basics Lightweight Super Soft Easy Care Microfiber Bed Sheet Set with Two Pillowcases - Twin, Bright White [B00U8QEXBS]


,retrieval_row_id,parent_asin,source,main_category,title,rating_number,average_rating,price,retrieval_score
0,100127,B07K8T7YHT,home_and_kitchen,Amazon Home,"Amazon Basics Kid's Soft Easy-Wash Lightweight Microfiber Sheet Set, Twin, Bright Aqua",64538,4.7,18.42,0.910746
1,101038,B0835K217P,home_and_kitchen,Amazon Home,"Amazon Basics Kid's Easy Care Microfiber Bed-in-a-Bag Bedding Set, Twin, 5-Piece Set, Multi-Color Dream Big",21575,4.8,40.00,0.871151
2,104199,B07P2FJ5V4,home_and_kitchen,Amazon Home,"Amazon Basics 6-Piece Ultra-Soft Microfiber Bed-In-A-Bag Comforter Bedding Set - Twin/Twin XL, Blue Watercolor",8690,4.6,38.29,0.864469
3,102285,B07P3K3DTZ,home_and_kitchen,Amazon Home,"Amazon Basics Comforter Set, Twin / Twin XL, Navy Blue, Microfiber, Ultra-Soft",13198,4.7,27.71,0.864337
4,101586,B07GB7RH9Q,home_and_kitchen,Amazon Home,"Amazon Basics Deluxe Microfiber Striped Sheet Set, Spa Blue,4 pieces, Queen",16756,4.6,26.79,0.863366



Anchor item (sports_and_outdoors): IRON °FLASK Sports Water Bottle - 32oz, 3 Lids (Straw Lid), Leak Proof - Stainless Steel Gym & Sport Bottles for Men, Women & Kids - Double Walled, Insulated Thermos, Metal Canteen [B08D9FWVJD]


,retrieval_row_id,parent_asin,source,main_category,title,rating_number,average_rating,price,retrieval_score
0,200154,B09PGRZ6NH,sports_and_outdoors,Tools & Home Improvement,"IRON °FLASK Sports Water Bottle - 40 Oz, 3 Lids (Spout Lid), Leak Proof, Vacuum Insulated Stainless Steel, Double Walled, Thermo Mug, Metal Canteen",23934,4.8,25.55,0.958371
1,253552,B09D6PVZCN,sports_and_outdoors,Amazon Home,"Water Bottle,Insulated Water Bottle 64 oz-3 Lids (Straw lid),Double Wall Insulated Water Jug,Anti-Slip Leakproof for Fitness,Gym and Outdoor Sports Thermos Water Bottle,BPA Free",298,4.7,27.98,0.894214
2,260035,B07H3P5YBY,sports_and_outdoors,Sports & Outdoors,"SIMPLE DRINK Stainless Steel Insulated Water Bottle | Wide Mouth Leak Proof Metal Water Bottle for Sport Travel, 1 or 3 Lids - Straw Lid, Flip Lid & Spout Lid (18oz, 32oz)",257,4.6,29.99,0.883668
3,295137,B0BZPG9TRG,sports_and_outdoors,Tools & Home Improvement,BOTTLE BOTTLE 32oz Insulated Water Bottle Stainless Steel Sport Water Bottle with Straw Dual-use Lid Design for Gym with Pill Box (gray),131,4.6,21.59,0.876900
4,216057,B08LPPXGYH,sports_and_outdoors,Amazon Home,"32 oz Basketball Water Bottle,Sports Flask with 2 Lids Straw Lid & Flex Cap,18/8 Stainless Steel Travel Tumbler Double Wall Vacuum Insulated Hot/Cold Gift for Mom Men (32oz, Basketball)",1126,4.8,19.99,0.873236


In [9]:
example_queries = [
    {"source": "electronics", "query": "wireless bluetooth headphones with noise cancelling"},
    {"source": "home_and_kitchen", "query": "nonstick frying pan for induction cooking"},
    {"source": "sports_and_outdoors", "query": "adjustable dumbbells for home gym workouts"},
]

for example in example_queries:
    print(f"\nQuery ({example['source']}): {example['query']}")
    display(hybrid_search(example["query"], top_k=5, source_filter=example["source"]))


Query (electronics): wireless bluetooth headphones with noise cancelling


,retrieval_row_id,parent_asin,source,main_category,title,rating_number,average_rating,price,retrieval_score
0,63969,B078KJJBGM,electronics,All Electronics,Sony WH-1000XM2/B Wireless Bluetooth Noise Cancelling Hi-Fi Headphones (Renewed),314,4.3,249.99,0.479551
1,4706,B0BH6V8TTG,electronics,All Electronics,Bose QuietComfort 45 Bluetooth Wireless Noise Cancelling Headphones - White Smoke,5647,4.7,329.00,0.478629
2,47817,B0B3LXYHVH,electronics,All Electronics,"Active Noise Cancelling Headphones,Wireless Bluetooth Headphones Built-in Mic 40 Hours Playtime Wireless Noise Cancelling Headphone 3D Low Bass Tone Fast Charge for Cellphone/Work/Gym/TravelComputers",470,4.3,39.31,0.478612
3,68296,B071JW71M9,electronics,All Electronics,Sony 950N1 Extra Bass Wireless Bluetooth Noise Cancelling Headphones - MDRXB950N1/B (Renewed),285,4.4,324.99,0.478039
4,16101,B082R9CSWR,electronics,Home Audio & Theater,AKG Noise Cancelling Headphones N60NC Wireless Bluetooth - Black - GP-N060HAHCAAA,1723,4.3,149.00,0.477638



Query (home_and_kitchen): nonstick frying pan for induction cooking


,retrieval_row_id,parent_asin,source,main_category,title,rating_number,average_rating,price,retrieval_score
0,197882,B09PNRH9PR,home_and_kitchen,Amazon Home,"Oursson Frying Pan Nonstick Induction, Flat Bottom, Cast Aluminum Stir Fry Pan with Non-Scratch Coating for Cooking, Sautee - Ideal for Gas, Electric, Induction & Ceramic Stoves (9.5 inch Pan)",509,4.6,19.98,0.481319
1,163932,B0C85KQFPF,home_and_kitchen,Amazon Home,"JEETEE Pots and Pans Set Nonstick 20PCS, Granite Coating Cookware Sets Induction Compatible with Frying Pan, Saucepan, Sauté Pan, Grill Pan, Cooking Pots, PFOA Free, (Grey, 20pcs Cookware Set)",826,4.7,199.99,0.479710
2,164021,B08PP73C14,home_and_kitchen,Amazon Home,"ANEDER Wok Pan Nonstick 12.5 Inch Skillet, Frying Pan with Lid & Spatula Wok Pans for Cooking Electric, Induction & Gas Stoves, Oven Safe",825,4.6,39.79,0.477799
3,142777,B09NQLH6LK,home_and_kitchen,Amazon Home,"WaxonWare 11 Inch Ceramic Nonstick Frying Pan/Nonstick Skillet, Large Anti-Warp Non-Toxic PTFE PFOA Free Granite Nonstick Pan, Induction Cooking Compatible, Non-Stick Egg Frying Pan- STONETEC Series",1249,4.3,43.95,0.477629
4,163005,B0C6SVWFMC,home_and_kitchen,Amazon Home,COOKER KING Nonstick Frying Pan with Lid-11Inch Nonstick Skillet Healthy Cookware＆PFOA Free Granite Coating | Cooking Pan with Heat-Resistant Handle-Induction Compatible,839,4.7,49.99,0.477253



Query (sports_and_outdoors): adjustable dumbbells for home gym workouts


,retrieval_row_id,parent_asin,source,main_category,title,rating_number,average_rating,price,retrieval_score
0,261610,B0C4YVL5K7,sports_and_outdoors,Sports & Outdoors,"Fiar Adjustable Weight Dumbbells Set- A Pair 4lb 6lb 8lb 10lb (2-5lb Each) Free Weights Set for Home Gym Equipment Workouts Strength Training for Women, Men,Teens 3 Colors",248,4.6,59.99,0.475838
1,269214,B0B2JWC8QD,sports_and_outdoors,Sports & Outdoors,"JX FITNESS Adjustable Dumbbells Hand Weights Set of 2-Neoprene Coated Exercise Fitness Dumbbells for Home Gym Equipment Workouts Strength Training Free Weights for Women, Men Seniors, Teens, and Y...",211,4.7,45.99,0.475243
2,296363,B08P542VWR,sports_and_outdoors,Sports & Outdoors,"Grip N Rip Fitness Adjustable Dumbbell Grip Converts Dumbbells with Handles Between 1-1.3"" into Kettlebells - Exercise Equipment for Home Workouts, the Gym, or On the Go - for Weights Up to 75 lbs",129,4.0,29.99,0.472677
3,295616,B000A6T9HO,sports_and_outdoors,Amazon Home,"POWERBLOCK Exercise Poster 3-Pack, 18” x 24” Each, 20-Minute Total Body Workout & 2 Dumbbell Workout Posters, Strength Training, Home Gym Posters, Use Adjustable Dumbbells",130,4.5,22.55,0.417442
4,296310,B08B5QS7KV,sports_and_outdoors,Sports & Outdoors,Adjustable Dumbbells Set 44 LB / 66 LB / 110 LB Home Weight Lifting Professional Dumbbells for Body Workout Home Gym Fitness with Carry Case,129,4.2,209.99,0.390092


## Baseline Limitations

This baseline is deliberately narrow. It captures lexical overlap in metadata, but it misses several things that matter in a conversational recommender:

- no user modeling or collaborative signal
- no parsing of explicit constraints such as budget, brand exclusions, or nuanced preferences
- no review-grounded evidence selection or explanation generation
- no reranking beyond lexical similarity plus simple rating-based tie-breaks
- a popularity-biased working catalog, because the baseline index is capped to a 45k-item subset for notebook safety

Those gaps define why the later notebooks add preference interpretation, stronger candidate filtering, review-grounded RAG, and reranking.
    

## Output Artifacts

The baseline produces lightweight artifacts so later notebooks can reload the index without rebuilding from scratch. `item_embeddings.npy` rows are aligned with `candidate_pool` row order after `reset_index(drop=True)` — the same alignment contract used by notebooks 04 and 05.

In [10]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

candidate_pool_path = ARTIFACT_DIR / "candidate_pool.parquet"
metadata_path = ARTIFACT_DIR / "index_metadata.json"

# candidate_pool — schema unchanged
candidate_pool_export = candidate_pool[
    [
        "retrieval_row_id",
        "parent_asin",
        "source",
        "main_category",
        "title",
        "store",
        "average_rating",
        "rating_number",
        "price",
        "retrieval_text",
        "retrieval_text_chars",
    ]
].copy()
con.register("candidate_pool_export", candidate_pool_export)
con.execute(
    f"COPY candidate_pool_export TO '{candidate_pool_path.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)"
)
con.unregister("candidate_pool_export")

# index metadata
metadata = {
    "baseline_name": BASELINE_NAME,
    "candidate_pool_strategy": {
        "items_per_source": ITEMS_PER_SOURCE,
        "max_pandas_rows": MAX_PANDAS_ROWS,
        "require_non_null_price": REQUIRE_NON_NULL_PRICE,
        "price_policy": PRICE_POLICY_DESCRIPTION,
        "selection_rule": "top items per source ordered by rating_number, average_rating, parent_asin",
    },
    "encoder_model": ENCODER_MODEL,
    "faiss_index_type": "IndexFlatIP",
    "embedding_dim": EMBEDDING_DIM,
    "bm25_tokenizer": "regex_alnum_lower",
    "rrf_k": RRF_K,
    "hybrid_pool_size": HYBRID_RETRIEVAL_POOL,
    "representation_text": "title + title + store + main_category + categories_path",
    "row_alignment": "Rows in item_embeddings.npy match candidate_pool.index order after reset_index(drop=True)",
}
metadata_path.write_text(json.dumps(metadata, indent=2))

# BM25, FAISS, and item_embeddings are saved incrementally in the index-building cell above.
bm25_index_path = ARTIFACT_DIR / "bm25_index.pkl"
faiss_index_path = ARTIFACT_DIR / "faiss_index.bin"
item_embeddings_path = ARTIFACT_DIR / "item_embeddings.npy"

artifact_summary = pd.DataFrame(
    [
        {"artifact": p.name, "size_mb": round(p.stat().st_size / (1024**2), 2)}
        for p in [candidate_pool_path, bm25_index_path, faiss_index_path, item_embeddings_path, metadata_path]
        if p.exists()
    ]
)
display(artifact_summary)

,artifact,size_mb
0,candidate_pool.parquet,38.78
1,bm25_index.pkl,81.95
2,faiss_index.bin,439.45
3,item_embeddings.npy,439.45
4,index_metadata.json,0.00
